# End-to-end test against Azure App Service

Tests the full v5.2 pipeline against the deployed Flask API.

**What this notebook does:**
1. Verifies the server is running v5.2 (fails fast if it's not)
2. Loads a local payload JSON file
3. POSTs to `/save`, gets a `load_id` back
4. Polls `/status/<load_id>` every 30 seconds (lightweight - no row data)
5. When complete, fetches `/results/<load_id>` once for the final output
6. Shows summary statistics and runs sanity checks

**Key properties:**
- Crash-safe polling - uses `clear_output(wait=True)`, won't spam the kernel
- Long polling timeout (2 hours by default) so you can leave it running
- Re-runnable - if you start a load and the notebook dies, run cell 5 again
  with the same `LOAD_ID` to resume polling the existing run

## 0. Config

In [ ]:
import json
import math
import time
from datetime import datetime, timezone
from pathlib import Path

import requests
from IPython.display import clear_output

# ----- Server -----
BASE_URL = 'https://desrapidev-fqaphqeaeuc7hafh.westcentralus-01.azurewebsites.net'

# ----- Input -----
PAYLOAD_PATH = Path(r'C:\\Users\\i40212888\\Downloads\\210-20260511T174006Z_payload.json')

# ----- Version verification -----
# Bumped to v5.2 - includes skip-empty-filters AND concurrent plan processing
EXPECTED_BUILD_VERSION = '2026-05-14-fast-pipeline-v5.2'

# ----- Polling -----
POLL_INTERVAL = 30      # seconds between /status checks
MAX_WAIT      = 7200    # 2 hour ceiling - more than enough for the new pipeline
POST_TIMEOUT  = 180     # POST timeout - large payloads need time to upload

# ----- Resume mode -----
# If you set this to a load_id from a previous run, the notebook skips the POST
# and just polls the existing run. Useful if the kernel died mid-run.
RESUME_LOAD_ID = None   # e.g. '210' to resume

print(f'Server:           {BASE_URL}')
print(f'Payload:          {PAYLOAD_PATH.name}')
print(f'Expected version: {EXPECTED_BUILD_VERSION}')
print(f'Resume load_id:   {RESUME_LOAD_ID or "(starting fresh)"}')


## 1. Health check - verify server is on v5.2

If this cell fails, stop and redeploy `build_benefits.py` + `checkpoint_runner.py`.
No point running a load against stale code.

In [ ]:
r = requests.get(f'{BASE_URL}/', timeout=15)
health = r.json()
print(json.dumps(health, indent=2))

server_ver = health.get('build_benefits_ver')
if server_ver != EXPECTED_BUILD_VERSION:
    raise SystemExit(
        f'\nSTOP - server is running {server_ver!r}, expected {EXPECTED_BUILD_VERSION!r}.\n'
        f'Redeploy build_benefits.py and checkpoint_runner.py, then restart App Service.'
    )

print(f'\nOK - server is on v5.2 with skip-empty + concurrent plans')


## 2. Load and sanitize the payload

Reads the local JSON file, strips out NaN/Inf values (JSON can't carry them),
and reports how many plans the payload contains.

In [ ]:
def sanitize(obj):
    '''Recursively replace NaN/Inf floats with None so JSON serialization works.'''
    if isinstance(obj, dict):
        return {k: sanitize(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [sanitize(v) for v in obj]
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
        return None
    return obj

raw = json.loads(PAYLOAD_PATH.read_text(encoding='utf-8'))
rows = raw['pbp'] if isinstance(raw, dict) and 'pbp' in raw else raw
clean = sanitize(rows)

input_plans = {r['FileName'].strip() for r in clean if r.get('FileName')}
input_load_ids = {str(r['LoadID']).strip() for r in clean if r.get('LoadID')}

print(f'PBP rows:        {len(clean):,}')
print(f'Distinct plans:  {len(input_plans)}')
print(f'LoadIDs:         {sorted(input_load_ids)}')

if len(input_load_ids) != 1:
    raise SystemExit(f'STOP - payload must have exactly one LoadID. Found: {input_load_ids}')

print(f'\nFirst 5 plan filenames:')
for fn in sorted(input_plans)[:5]:
    print(f'  - {fn}')


## 3. POST the payload (skipped if RESUME_LOAD_ID is set)

Returns a `load_id` in 2-5 seconds. Processing continues in the background
on the server.

In [ ]:
if RESUME_LOAD_ID:
    load_id = RESUME_LOAD_ID
    print(f'Resuming poll of load_id={load_id} (skipping POST)')
    post_time = 0.0
else:
    t_post = time.monotonic()
    r = requests.post(
        f'{BASE_URL}/save',
        json=clean,
        headers={'Content-Type': 'application/json'},
        timeout=POST_TIMEOUT,
    )
    post_time = time.monotonic() - t_post

    if r.status_code != 202:
        raise RuntimeError(f'POST failed: {r.status_code} {r.text[:500]}')

    resp = r.json()
    load_id = resp['load_id']

    print(f'POST: {r.status_code} in {post_time:.1f}s')
    print(json.dumps(resp, indent=2))


## 4. Poll /status until done

Uses the lightweight `/status` endpoint - returns plan counts only, not
the row data. Refreshes a single output line every 30s using
`clear_output` so the notebook doesn't accumulate hundreds of lines.

Safe to interrupt and re-run with `RESUME_LOAD_ID` set.

In [ ]:
t_proc = time.monotonic()
elapsed = 0
final_status = None
last_n_done = -1
stall_count = 0

print(f'Polling /status/{load_id} every {POLL_INTERVAL}s (max wait: {MAX_WAIT}s)')
print('Open the App Service log stream to see per-plan progress in real time.\n')

while elapsed < MAX_WAIT:
    try:
        r = requests.get(f'{BASE_URL}/status/{load_id}', timeout=30)
        if r.status_code == 404:
            clear_output(wait=True)
            print(f'No status blob for {load_id} yet - was it just submitted?')
            time.sleep(POLL_INTERVAL)
            elapsed += POLL_INTERVAL
            continue

        s = r.json()
        n_done = s.get('n_plans_done', 0)
        n_total = s.get('n_plans_total', '?')
        n_failed = s.get('n_plans_failed', 0)
        status = s.get('status')

        # Track stall detection
        if n_done == last_n_done:
            stall_count += 1
        else:
            stall_count = 0
            last_n_done = n_done

        # Compute rate + ETA
        rate_info = ''
        if n_done > 0 and isinstance(n_total, int):
            avg_per_plan = elapsed / n_done if n_done else 0
            remaining = (n_total - n_done) * avg_per_plan
            rate_info = f'\n  avg/plan: {avg_per_plan:.1f}s | ETA: {remaining:.0f}s ({remaining/60:.1f} min)'

        clear_output(wait=True)
        print(f'[{datetime.now().strftime("%H:%M:%S")}]  load_id={load_id}')
        print(f'  status:       {status}')
        print(f'  plans done:   {n_done}/{n_total}')
        print(f'  plans failed: {n_failed}')
        print(f'  updated_at:   {s.get("updated_at")}')
        print(f'  elapsed:      {elapsed}s ({elapsed/60:.1f} min){rate_info}')

        if stall_count >= 10:
            print(f'\n  WARNING: n_plans_done has not changed in {stall_count * POLL_INTERVAL}s')
            print(f'  Worker may have died. Check App Service logs.')

        if status in ('success', 'partial', 'error'):
            final_status = s
            break

    except requests.exceptions.ReadTimeout:
        print(f'  (poll timeout at {elapsed}s, retrying)')
    except Exception as e:
        print(f'  (poll error at {elapsed}s: {e})')

    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL

proc_time = time.monotonic() - t_proc

if not final_status:
    print(f'\nTimed out after {MAX_WAIT}s. Last seen status:')
    print(json.dumps(s, indent=2))
    raise SystemExit('Stop - no completion within MAX_WAIT. Re-run with RESUME_LOAD_ID set.')

print(f'\n--- FINAL STATUS ---')
print(f'  status:        {final_status.get("status")}')
print(f'  plans done:    {final_status.get("n_plans_done")}/{final_status.get("n_plans_total")}')
print(f'  plans failed:  {final_status.get("n_plans_failed")}')
print(f'  processing:    {proc_time:.1f}s ({proc_time/60:.1f} min)')
print(f'  build:         {final_status.get("build_version")}')


## 5. Fetch the final combined results

One large `/results` call to pull the full row data.

In [ ]:
print(f'Fetching /results/{load_id} (this can be large - hundreds of KB to several MB)')
t_fetch = time.monotonic()
r = requests.get(f'{BASE_URL}/results/{load_id}', timeout=120)
fetch_time = time.monotonic() - t_fetch

if r.status_code not in (200, 202):
    raise RuntimeError(f'Results fetch failed: {r.status_code} {r.text[:500]}')

result_data = r.json()
output_rows = result_data.get('results', [])

print(f'Fetched in {fetch_time:.1f}s')
print(f'Output:')
print(f'  result_count:   {result_data.get("result_count")}')
print(f'  plan_count:     {result_data.get("plan_count")}')
print(f'  output_blob:    {result_data.get("output_blob")}')

total_time = post_time + proc_time + fetch_time
print(f'\nTotal wall time: {total_time:.1f}s ({total_time/60:.1f} min)')
print(f'   - POST:        {post_time:.1f}s')
print(f'   - Processing:  {proc_time:.1f}s')
print(f'   - Fetch:       {fetch_time:.1f}s')


## 6. Sanity checks on output

Make sure every input plan produced output, count rows per benefit, spot
anything obviously broken.

In [ ]:
import pandas as pd

if not output_rows:
    print('No rows in output - cannot run sanity checks')
else:
    df = pd.DataFrame(output_rows)

    print(f'Total rows:           {len(df):,}')
    print(f'Distinct plans:       {df["planid"].nunique()}')
    print(f'Distinct benefit IDs: {df["benefitid"].nunique()}')

    # Coverage check - every input plan should have output
    output_plans = set(df['planid'].unique())
    missing_plans = []
    for input_fn in input_plans:
        # Strip the carrier prefix and reformat to match the planid format
        # FileName: "HCSC_H0107-003-000"  ->  planid: "HCSC_H0107_003_0"
        parts = input_fn.split('_', 1)
        if len(parts) == 2:
            contract_plan_seg = parts[1].split('-')
            if len(contract_plan_seg) >= 3:
                segment = contract_plan_seg[2].lstrip('0') or '0'
                expected = f'{parts[0]}_{contract_plan_seg[0]}_{contract_plan_seg[1]}_{segment}'
                if not any(pid.startswith(parts[0]) and contract_plan_seg[0] in pid and contract_plan_seg[1] in pid
                           for pid in output_plans):
                    missing_plans.append(input_fn)

    if missing_plans:
        print(f'\nWARNING: {len(missing_plans)} input plan(s) had no output:')
        for fn in missing_plans[:10]:
            print(f'  - {fn}')
    else:
        print(f'\nAll {len(input_plans)} input plans produced output')


### Rows per plan (top 20)

In [ ]:
if not df.empty:
    rows_per_plan = df.groupby('planid').size().sort_values(ascending=False)
    print(f'Rows per plan (top 20 of {len(rows_per_plan)}):')
    print(rows_per_plan.head(20).to_string())
    print()
    print(f'Min:    {rows_per_plan.min()}')
    print(f'Max:    {rows_per_plan.max()}')
    print(f'Median: {rows_per_plan.median():.0f}')
    print(f'Mean:   {rows_per_plan.mean():.1f}')


### Rows per benefit ID (all)

In [ ]:
if not df.empty:
    df['benefitid'] = df['benefitid'].astype(str)
    rows_per_bid = df.groupby('benefitid').size().sort_values(ascending=False)
    plans_per_bid = df.groupby('benefitid')['planid'].nunique()

    summary = pd.DataFrame({
        'total_rows':   rows_per_bid,
        'plans_w_data': plans_per_bid,
        'pct_plans':    (100 * plans_per_bid / df['planid'].nunique()).round(0).astype(str) + '%',
    }).sort_values('total_rows', ascending=False)

    print(f'All benefit IDs in output:')
    print(summary.to_string())


## 7. Per-plan timing breakdown

From the status blob - shows which plans were fast vs slow. Useful for
spotting outliers.

In [ ]:
plans_status = final_status.get('plans', {})
timing_rows = []
for fn, info in plans_status.items():
    if info.get('status') == 'done':
        timing_rows.append({
            'filename':  fn,
            'rows':      info.get('rows', 0),
            'elapsed_s': info.get('elapsed_s', 0),
        })

if timing_rows:
    timing_df = pd.DataFrame(timing_rows).sort_values('elapsed_s', ascending=False)
    print(f'Slowest 15 plans:')
    print(timing_df.head(15).to_string(index=False))
    print()
    print(f'Per-plan stats:')
    print(f'  fastest:  {timing_df["elapsed_s"].min():.1f}s')
    print(f'  slowest:  {timing_df["elapsed_s"].max():.1f}s')
    print(f'  median:   {timing_df["elapsed_s"].median():.1f}s')
    print(f'  mean:     {timing_df["elapsed_s"].mean():.1f}s')
    print(f'  total:    {timing_df["elapsed_s"].sum():.1f}s (sum of per-plan times)')
    print(f'  wall:     {proc_time:.1f}s')
    speedup = timing_df['elapsed_s'].sum() / proc_time if proc_time else 0
    print(f'  parallel speedup factor: {speedup:.2f}x')

failed = [(fn, info) for fn, info in plans_status.items() if info.get('status') == 'failed']
if failed:
    print(f'\nFailed plans: {len(failed)}')
    for fn, info in failed[:10]:
        print(f'  {fn}: {info.get("error", "")[:120]}')


## 8. Save output locally (optional)

Useful if you want to compare against a previous run or hand off to Morteza.

In [ ]:
local_out = Path.cwd() / f'output_benefits_{load_id}.json'
local_out.write_text(json.dumps(output_rows, indent=2), encoding='utf-8')
print(f'Saved {len(output_rows):,} rows to {local_out}')
print(f'File size: {local_out.stat().st_size:,} bytes ({local_out.stat().st_size/1024:.1f} KB)')


## 9. Verdict

One-paragraph summary suitable for sharing in a status update.

In [ ]:
print('=' * 70)
print(f'Run summary: load_id={load_id}')
print('=' * 70)
print(f'Build version:        {final_status.get("build_version")}')
print(f'Plans:                {final_status.get("n_plans_done")}/{final_status.get("n_plans_total")} done | '
      f'{final_status.get("n_plans_failed")} failed')
print(f'Output rows:          {len(output_rows):,}')
print(f'Total wall time:      {total_time/60:.1f} min')
if timing_rows:
    print(f'Sum of per-plan time: {timing_df["elapsed_s"].sum()/60:.1f} min')
    print(f'Parallel speedup:     {speedup:.2f}x')
print()
if final_status.get('status') == 'success':
    print('Status: SUCCESS - all plans complete, output available at:')
    print(f'  {final_status.get("output_blob")}  (in outbound container)')
    print(f'  {local_out}  (local copy)')
elif final_status.get('status') == 'partial':
    pending = final_status.get('n_plans_total', 0) - final_status.get('n_plans_done', 0) - final_status.get('n_plans_failed', 0)
    print(f'Status: PARTIAL - {pending} plan(s) pending, {final_status.get("n_plans_failed")} failed.')
    print(f'Action: re-POST the same payload (existing checkpoints will be reused),')
    print(f'        OR re-run this notebook with RESUME_LOAD_ID = "{load_id}"')
else:
    print(f'Status: {final_status.get("status")}')
